## 1. 필요 라이브러리 호출

In [1]:
# 환경설정
import os
import sys
import time
from tqdm import tqdm
import nest_asyncio
nest_asyncio.apply()
from dotenv import load_dotenv

load_dotenv()
# duckdb
import duckdb

# 데이터 전처리
import re
import pandas as pd
import numpy as np
import polars as pl
from datetime import datetime, timedelta
from copy import deepcopy

# 데이터 수집
import requests
from bs4 import BeautifulSoup


# VectorDB 저장
from hashlib import md5
from langchain_community.vectorstores.utils import filter_complex_metadata # ChromaDB가 제공하지 못하는 데이터 형태를 자동으로 string처리
from datetime import datetime, timezone

## LLM 활용
from summary_function import NewsSummaryAgent
# LLM 활용을 위한 dict형태 구축
from collections import defaultdict

# langchain 계열
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

from langchain_openai import ChatOpenAI

# 1. LLM 모델 세팅 (OpenAI)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
#from summary_function_openai import NewsSummaryAgent, build_summary_graph  # 너가 만든 것
from summart_function_openai_2 import NewsSummaryAgent, build_summary_graph  # 너가 만든 것
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.3)

# ㄱRe-ranker 모델 활용
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder


## 2. ETF 목록 가져오기

In [2]:
ETF_conn = duckdb.connect('../DB/ETF.db')

In [3]:
ETF_df = ETF_conn.execute('select * from IRP_ETF_COMPOSE_table').fetchdf()
ETF_conn.close()

In [4]:
ETF_df = ETF_df.map(lambda x : x.strip())

In [5]:
ETF_df_task_1 = ETF_df[ETF_df['구성종목 종목명'] != "설정현금액"]
ETF_df_task_1 = ETF_df_task_1[ETF_df_task_1['구성종목 종목명'] != "원화현금"]
ETF_df_task_1 = ETF_df_task_1[1:]

In [6]:
# 원하는 컬럼 필터링
ETF_df_task_2 = ETF_df_task_1[['ETF 종목명','구성종목 표준코드','구성종목 종목명','편입비율']]

In [7]:
# 구성종목 중 상위 5개 추출
ETF_df_task_3 = ETF_df_task_2.sort_values(by=['ETF 종목명','편입비율'], ascending=False)

In [8]:
# 종목별 상위 5개
ETF_df_task_4= ETF_df_task_3.groupby('ETF 종목명').head(5)

In [9]:
ETF_df_task_4[ETF_df_task_4['ETF 종목명'].str.contains('반도체')]
ETF_df_task_4[ETF_df_task_4['ETF 종목명'].str.contains('전지')]
ETF_df_task_4[ETF_df_task_4['ETF 종목명'].str.contains('자율주행')]
ETF_df_task_4[ETF_df_task_4['ETF 종목명'].str.contains('금융')]

,ETF 종목명,구성종목 표준코드,구성종목 종목명,편입비율
34483,TIGER 25-12 금융채(AA-이상),KR6205498EC7,하나카드273,4.988738
34398,TIGER 25-12 금융채(AA-이상),KR6005273F23,아이엠뱅크46-02이12A-21,4.10297
34473,TIGER 25-12 금융채(AA-이상),KR6140178EB5,케이비국민카드421-1,3.300191
34461,TIGER 25-12 금융채(AA-이상),KR6079314EA3,JB 우리캐피탈524-1(지),2.485386
34475,TIGER 25-12 금융채(AA-이상),KR6145763DC9,BNK캐피탈338-3,2.48222
7606,TIGER 200 금융,KR7316140003,우리금융지주,7.825771
7587,TIGER 200 금융,KR7000810002,삼성화재,7.048532
7595,TIGER 200 금융,KR7032830002,삼성생명,5.727802
7607,TIGER 200 금융,KR7323410001,카카오뱅크,5.180046
7602,TIGER 200 금융,KR7138040001,메리츠금융지주,4.741424


## 3. 고객 데이터 시나리오
 - 고객 데이터 생성

In [10]:
Customer_A = ETF_df_task_4[ETF_df_task_4['ETF 종목명'].isin(['ACE AI반도체포커스','ACE 2차전지&친환경차액티브','KODEX 자율주행액티브','RISE 200금융'])]

In [11]:
Customer_A = Customer_A.reset_index().drop('index',axis = 1)

In [12]:
Custer_Having_ticker_lst = list(Customer_A['구성종목 종목명'].unique())

In [13]:
Custer_Having_ticker_lst

['우리금융지주',
 '삼성화재',
 '삼성생명',
 '카카오뱅크',
 '메리츠금융지주',
 '현대모비스',
 '현대오토에버',
 'SK하이닉스',
 '현대글로비스',
 '현대차',
 '삼성전자',
 '한미반도체',
 '파크시스템스',
 'DB하이텍',
 '기아',
 'POSCO홀딩스',
 'LG에너지솔루션']

## 4. 고객 데이터 저장

In [14]:
con = duckdb.connect('../DB/Customer.db')

# Pandas DataFrame을 DuckDB에서 참조할 수 있도록 등록
con.register('temp_df', Customer_A)

# 테이블이 없다면 생성
con.execute("""
    CREATE TABLE IF NOT EXISTS Customers AS
    SELECT * FROM temp_df LIMIT 0
""")

# 데이터 삽입
con.execute("INSERT INTO Customers SELECT * FROM temp_df")

# 정리
con.unregister('temp_df')
con.close()

## 5. 고객이 보유하고 있는 데이터를 DB에 저장하기

In [77]:
import asyncio
import aiohttp
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin,urlparse
import duckdb

# ▶ 헤더
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36"
}

# ▶ 기사 리스트 파싱 함수
async def get_article_urls(session, page_url):
    try:
        async with session.get(page_url, headers=headers) as resp:
            text = await resp.text()
            soup = BeautifulSoup(text, 'html.parser')
            ul = soup.select_one('#content > div.left_cont > div > div.section.hk_news > div.section_cont > ul')
            if not ul:
                return []

            urls = []
            for a in ul.find_all('a', href=True):
                href = a['href']
                if '/article/' in href:
                    urls.append(href)
            return list(set(urls))  # 중복 제거
    except Exception as e:
        print(f"[get_article_urls error] {page_url} - {e}")
        return []

# ▶ 기사 상세 파싱 함수
async def fetch_article(session, url):
    try:
        async with session.get(url, headers=headers) as resp:
            html = await resp.text()
            soup = BeautifulSoup(html, 'html.parser')

            hostname = urlparse(url).hostname

            # ✅ 1. magazine.hankyung.com용 로직
            if 'magazine.hankyung.com' in hostname:
                return {
                    'header': soup.select_one('#contents h1.news-tit').text.strip() if soup.select_one('#contents h1.news-tit') else None,
                    'summary': None,
                    'content': soup.select_one('#magazineView').text.strip() if soup.select_one('#magazineView') else None,
                    'url': url,
                    'datetime': soup.select_one('#contents span.txt-num').text.strip() if soup.select_one('#contents span.txt-num') else None,
                }

            # ✅ 2. www.hankyung.com일 경우 기존 로직
            elif 'hankyung.com' in hostname:
                return {
                    'header': soup.select_one('h1.headline').text.strip() if soup.select_one('h1.headline') else None,
                    'summary': soup.select_one('div.summary').text.strip() if soup.select_one('div.summary') else None,
                    'content': soup.select_one('#articletxt').text.strip() if soup.select_one('#articletxt') else None,
                    'url': url,
                    'datetime': soup.select_one('div.datetime span.txt-date').text.strip() if soup.select_one('div.datetime span.txt-date') else None,
                }

            # ✅ 알 수 없는 도메인
            else:
                print(f"⚠️ 알 수 없는 호스트: {hostname}")
                return {'url': url, 'header': None, 'summary': None, 'content': None, 'datetime': None}

    except Exception as e:
        print(f"[fetch_article error] {url} - {e}")
        return {'url': url, 'header': None, 'summary': None, 'content': None, 'datetime': None}
# ▶ 메인 비동기 루프
async def extract_news_data_async(query_text, page_range):
    base_url = 'https://search.hankyung.com/search/news?query={query}&page={page}'
    search_urls = [base_url.format(query=query_text, page=p+1) for p in range(page_range)]

    async with aiohttp.ClientSession() as session:
        # 1. 페이지별 기사 링크 수집
        tasks = [get_article_urls(session, url) for url in search_urls]
        results = await asyncio.gather(*tasks)
        article_urls = list(set([url for sublist in results for url in sublist]))

        print(f"🔗 총 {len(article_urls)}개의 기사 URL 수집됨")

        # 2. 기사 본문 수집
        article_tasks = [fetch_article(session, url) for url in article_urls]
        articles = await asyncio.gather(*article_tasks)

        # 3. ticker 컬럼 추가
        for article in articles:
            article['ticker'] = query_text

        # 4. 비어 있으면 dummy row 추가
        if not articles:
            articles = [{
                'header': None,
                'summary': None,
                'content': None,
                'url': None,
                'datetime': None,
                'ticker': query_text
            }]
            print("⚠️ 수집된 기사가 없어 None 값으로 대체 저장합니다.")

        # 4. DuckDB 저장
        df = pd.DataFrame(articles)
        con = duckdb.connect('../DB/Customer_news.db')

        # Pandas DataFrame을 DuckDB에서 참조할 수 있도록 등록
        con.register('temp_df', df)

        # 테이블이 없다면 생성
        con.execute("""
            CREATE TABLE IF NOT EXISTS articles AS
            SELECT * FROM temp_df LIMIT 0
        """)

        # 데이터 삽입
        con.execute("INSERT INTO articles SELECT * FROM temp_df")

        # 정리
        con.unregister('temp_df')
        con.close()

        print(f"✅ 저장 완료: ../DB/Customer_news.db (ticker = {query_text})")

# ▶ 실행 함수
def extract_news_data(query_text, page_range):
    loop = asyncio.get_event_loop()
    loop.run_until_complete(extract_news_data_async(query_text, page_range))

In [ ]:
for ticker in Custer_Having_ticker_lst:
    extract_news_data(ticker,50)

🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 우리금융지주)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 삼성화재)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 삼성생명)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 카카오뱅크)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 메리츠금융지주)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 현대모비스)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 현대오토에버)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = SK하이닉스)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 현대글로비스)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 현대차)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = SK하이닉스)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 삼성전자)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 한미반도체)
🔗 총 500개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 파크시스템스)
🔗 총 500개의 기사 URL 

저장이 잘 되었는 지 확인

In [15]:
cus_news = duckdb.connect('../DB/Customer_news.db')

In [16]:
cusA_news_df = cus_news.execute('select * from articles').fetch_df()
cus_news.close()

In [17]:
cusA_news_df.head(5)

,header,summary,content,url,datetime,ticker
0,"18일, 거래소 외국인 순매수상위에 전기,전자 업종 4종목",None,"외국인 투자자는 18일 거래소에서 삼성전자, NAVER, 크래프톤 등을 중점적으로 ...",https://www.hankyung.com/article/202506184343L,2025.06.18 18:35,우리금융지주
1,"""실적·주주환원 훈풍""…은행·증권주 신고가 행진, 스탁론 매수세도 유입",None,국내 은행 및 증권주들이 2분기 실적 호조와 주주환원 기대감에 힘입어 강세를 이어가...,https://www.hankyung.com/article/202507096279a,2025.07.09 10:30,우리금융지주
2,"12일, 외국인 거래소에서 한화에어로스페이스(+5.3%), 현대차(+0.25%) 등...",None,"외국인 투자자는 12일 거래소에서 한화에어로스페이스, 현대차, 현대건설 등을 중점적...",https://www.hankyung.com/article/202506121791L,2025.06.12 18:35,우리금융지주
3,"금감원, 우리금융 경평 '2→3등급' 결론…이번주 통보할 듯",None,금감원 / 사진=노정동 기자\n\n 금융감독원이 우리금융...,https://www.hankyung.com/article/2025031784556,2025.03.17 11:18,우리금융지주
4,한도 초과 걱정 없이 연 4%대 금리로 저점 집중투자 시작하기!,None,부자네스탁론이 특별 이벤트로 5년고정 연 4.9%의 저금리 스탁론 상품을 출시하면서...,https://www.hankyung.com/article/202506056123a,2025.06.05 14:38,우리금융지주


In [18]:
cusA_news_df.shape

(10000, 6)

In [19]:
# 중복제거 필요...
cusA_news_df.groupby('ticker').count()

,header,summary,content,url,datetime
ticker,,,,,
DB하이텍,500,62,500,500,500
LG에너지솔루션,500,96,500,500,500
POSCO홀딩스,500,25,500,500,500
SK하이닉스,1000,238,1000,1000,1000
기아,499,238,499,500,499
메리츠금융지주,500,128,500,500,500
삼성생명,500,165,500,500,500
삼성전자,500,161,500,500,500
삼성화재,499,186,499,500,499


In [20]:
cusA_news_df = cusA_news_df.drop_duplicates()

In [21]:
cusA_news_df.groupby('ticker').count()

,header,summary,content,url,datetime
ticker,,,,,
DB하이텍,500,62,500,500,500
LG에너지솔루션,500,96,500,500,500
POSCO홀딩스,500,25,500,500,500
SK하이닉스,500,119,500,500,500
기아,499,238,499,500,499
메리츠금융지주,500,128,500,500,500
삼성생명,500,165,500,500,500
삼성전자,500,161,500,500,500
삼성화재,499,186,499,500,499


## 번외) 날짜 전처리 확인 : 최근 N일치 가져오는 로직 생성

In [22]:
cusA_news_df['Date'] = cusA_news_df.datetime.str[:10]
cusA_news_df = cusA_news_df.drop('datetime',axis=1)
cusA_news_df['Date'] = pd.to_datetime(cusA_news_df['Date'])

In [23]:
# 2. 기준일 계산 (오늘 날짜 - 5일)
today = pd.to_datetime("2025-07-30")
five_days_ago = today - timedelta(days=30)

# 3. 최근 5일치 필터링
recent_news_df = cusA_news_df[cusA_news_df['Date'] >= five_days_ago]

In [24]:
recent_news_df

,header,summary,content,url,ticker,Date
1,"""실적·주주환원 훈풍""…은행·증권주 신고가 행진, 스탁론 매수세도 유입",None,국내 은행 및 증권주들이 2분기 실적 호조와 주주환원 기대감에 힘입어 강세를 이어가...,https://www.hankyung.com/article/202507096279a,우리금융지주,2025-07-09
5,"미래에셋만 너무 비싸다?…""PBR 1.2배도 가능""",None,영상 모듈 닫기\n\n\n\n\n<앵커> 증시 상승세에 힘입어 국내 증권사들도 2분...,https://www.hankyung.com/article/2025071501385,우리금융지주,2025-07-15
7,"09일, 외국인 거래소에서 삼성전자(-1.63%), 두산에너빌리티(-3.3%) 등 순매도",None,"외국인 투자자는 09일 거래소에서 삼성전자, 두산에너빌리티, 삼성SDI 등을 중점적...",https://www.hankyung.com/article/202507098435L,우리금융지주,2025-07-09
8,"01일, 코스닥 외국인 순매수상위에 일반전기전자 업종 5종목",None,"외국인 투자자는 01일 코스닥에서 리가켐바이오, 솔브레인, 디앤디파마텍 등을 중점적...",https://www.hankyung.com/article/202508017025L,우리금융지주,2025-08-01
13,반도체 대장주 자리 꿰차더니…SK하이닉스 개미들 '두근두근',2분기 실적 시즌 돌입…영업이익 추정치 살펴보니\n\n하이닉스 영업익 첫 9조 넘을...,2분기 실적 발표 시즌에 본격 돌입하면서 주도주의 성적표가 윤곽을 드러내고 있다. ...,https://www.hankyung.com/article/2025072252001,우리금융지주,2025-07-22
...,...,...,...,...,...,...
9995,"코스피, 3200선 회복…테슬라 업은 삼성전자, 7만원대 탈환",코스닥은 0.3% '하락'\n원·달러 환율 1382원에 주간거래 마쳐,28일 서울 중구 하나은행 본점 딜링룸에서 직원들이 업무를 보고 있다. /사진=연합...,https://www.hankyung.com/article/2025072866196,LG에너지솔루션,2025-07-28
9996,"코스피, 3년10개월 만에 3200선 돌파…'연고점 또 경신'",삼성전자·하이닉스 2%대 강세,사진=연합뉴스\n\n 코스피지수가 11일 개인투자자의 매...,https://www.hankyung.com/article/2025071119236,LG에너지솔루션,2025-07-11
9997,"코스피, 3190선 약세 출발…코스닥은 강보합",None,전날인 14일 오후 서울 중구 하나은행 본점 딜링룸 전광판. /사진=뉴스1\n\n ...,https://www.hankyung.com/article/2025071582926,LG에너지솔루션,2025-07-15
9998,"이자 부담은 최소로, 투자 효율은 최대로! 신용대출 3%대 활용법",None,"전송종목 : 바이오비쥬, 엘브이엠씨홀딩스, 크리스탈신소재, GRT, 잉글우드랩최근 ...",https://www.hankyung.com/article/202507071499a,LG에너지솔루션,2025-07-07


In [25]:
recent_news_df= recent_news_df.drop_duplicates()

In [26]:
# 최근 30일치 뉴스 기사 필터링 시 남아있는 뉴스 기사 개수
recent_news_df.groupby('ticker').count()

,header,summary,content,url,Date
ticker,,,,,
DB하이텍,30,4,30,30,30
LG에너지솔루션,500,96,500,500,500
POSCO홀딩스,464,23,464,464,464
SK하이닉스,500,119,500,500,500
기아,411,191,411,411,411
메리츠금융지주,51,5,51,51,51
삼성생명,140,45,140,140,140
삼성전자,500,161,500,500,500
삼성화재,96,31,96,96,96


## 정규식으로 뉴스 기사 내 불용어 처리

In [27]:
boilerplate_patterns = [
    r"\* 아래 텍스트는 실제 방송 내용과 차이가 있을 수 있으니.*",
    r"\*인터뷰를 인용보도할 때는 프로그램명.*",
    r"저작권은.*에 있습니다.*",
    r"▶ 알립니다.*",
    r"\[앵커\].*?\[",
    r"\[기자\].*?\[",
    r"\[.*?\]",             # 모든 대괄호 안 내용
    r"영상취재:.*",
    r"영상편집:.*",
    r"그래픽:.*",
    r"\/?사진=(연합뉴스|뉴스\S*)[^\n]*\s*"       # 사진=출처 or /사진=출처 패턴 제거
]

In [28]:
compiled_pattern = re.compile("|".join(boilerplate_patterns))

In [29]:
test_pattern = cusA_news_df.content.astype(str).apply(lambda x : compiled_pattern.sub("", x).strip())

In [30]:
cusA_news_df.content.astype(str).apply(lambda x : len(x)).describe()

count      8500.000000
mean       1677.332000
std        7343.833444
min           0.000000
25%         707.000000
50%        1077.000000
75%        1496.000000
max      125274.000000
Name: content, dtype: float64

In [31]:
test_pattern.apply(lambda x : len(x)).describe()

count      8500.000000
mean       1660.115294
std        7342.442298
min           0.000000
25%         689.750000
50%        1073.000000
75%        1496.000000
max      125274.000000
Name: content, dtype: float64

#### 결론 : 큰 영향 없다..? 그냥 해보자

## 질문 생성 (For Labeling)  -- 주말 작업 예정 .. 모델 성능평가를 위함
Part_1 : Re-Ranking 라벨링

In [32]:
def labeling(data):
  template = """
  <instruction>
  다음은 뉴스 기사 본문과 해당 기사에 매핑된 종목명(티커)입니다.
  당신의 임무는 기사 본문이 해당 종목에 대한 기사인지 여부를 판별하는 것입니다.

  규칙:
  - 1 : 뉴스 내용이 해당 종목 기사 내용임.
     예: 종목명이 기사에 등장하고, 그 종목의 실적, 주가, 제품, 사건, 경영, 산업 동향 등과 밀접한 관련이 있음.
     예) 종목의 실적/주가/사업/이슈/계약/정책/규제/소송/리스크/전망 등.
     예) 섹터 기사라도 해당 종목이 사례/주요 구성원으로 명시적 언급되고 맥락에 기여.
  - 0 : 뉴스 내용이 해당 종목과 직접적으로 관련이 없음  
     예: 종목명이 전혀 등장하지 않거나, 비슷한 용어를 가진 단어의 내용이 등장하더라도 다른 주제가 메인인 경우.
     예) 피상적 나열(태그/키워드/꼬리말 광고)만 존재, 타사 이슈가 중심.
     
  출력 형식:
  - 숫자 1 또는 0만 출력
  </instruction>

  예시:
  ---
  [티커] 삼성전자
  [본문] 삼성전자가 2분기 실적 호조를 발표하며 주가가 3% 상승했다.
  [정답] 1
  ---
  [티커] 삼성전자
  [본문] 미국 증시가 기술주 중심으로 상승세를 보였다. 애플과 구글 주가가 상승했다.
  [정답] 0
  ---

  다음 데이터를 분류하세요.

  [티커] {ticker_name}
  [본문] {news_content}
  [정답]
  """

  prompt = ChatPromptTemplate.from_template(template)

  # 3. 체인 구성
  chain = prompt | llm | StrOutputParser()

  # 4. 실행 예시
  query = chain.invoke({"news_content": data,'ticker_name' : '우리금융지주'})
  return query

## 6. VectorDB 저장

In [33]:
# 4. OpenAI 임베딩 모델 로딩
embedding = OpenAIEmbeddings(model="text-embedding-3-large")

persist_directory = "../VectorDB/chroma_news_db"

vectordb = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding,
    collection_name="SLM_News")

In [34]:
def pick_splitter_by_length(text_len: int) -> RecursiveCharacterTextSplitter:
    """
    뉴스 본문의 길이에 따라 적절한 텍스트 분할기를 반환합니다.
    """
    if text_len <= 1200:
        # 짧은 기사 → 굳이 자르지 않고 1덩어리로 처리
        return RecursiveCharacterTextSplitter(
            chunk_size=1200,
            chunk_overlap=0,
            separators=["\n\n", "\n", " ", ""]
        )
    elif text_len <= 10_000:
        # 중간 길이 → 일반적인 1,200자 기준으로 분할
        return RecursiveCharacterTextSplitter(
            chunk_size=1200,
            chunk_overlap=150,
            separators=["\n\n", "\n", " ", ""]
        )
    elif text_len <= 50_000:
        # 긴 기사 → 덩어리를 좀 더 키움
        return RecursiveCharacterTextSplitter(
            chunk_size=1800,
            chunk_overlap=200,
            separators=["\n\n", "\n", " ", ""]
        )
    else:
        # 초장문 → 더 크게 자르되, 요약도 고려 (이건 후속 처리 필요)
        return RecursiveCharacterTextSplitter(
            chunk_size=2000,
            chunk_overlap=200,
            separators=["\n\n", "\n", " ", ""]
        )
    

In [35]:
# 2. 문서 리스트 생성 (chunk + metadata 포함)
def make_documents(df):
    docs = []

    for idx, row in tqdm(df.iterrows()):
        text = row["content"]
        splitter = pick_splitter_by_length(len(text))
        chunks = splitter.split_text(text)
        for i, chunk in enumerate(chunks):
            label = labeling(chunks)
            metadata = {
                "title": row["header"],
                "url": row["url"],
                "Date": row["Date"],
                "ticker": row.get("ticker", "None"),
                "chunk_idx": i,
                "original_idx": idx,
                'label' : label  #labeling한 결과를 넣자!! 
            }
            time.sleep(0.1)
            docs.append(Document(page_content=chunk, metadata=metadata))

    return docs


In [36]:
def get_recent_articles(df: pd.DataFrame, ticker: str, days: int = 5):
    df['Date'] = pd.to_datetime(df['Date'])
    today = df['Date'].max() # 가지고 있는 뉴스의 가장 최신 데이터
    recent_df = df[
        (df['ticker'] == ticker) &
        (df['Date'] >= today - timedelta(days=days))
    ]
    return recent_df.sort_values(by="Date", ascending=False)


In [37]:
def make_doc_id(d: Document) -> str:
    """
    url + chunk_idx(없으면 0) + 시간
    """
    run_utc = datetime.now(timezone.utc)
    base = f"{d.metadata.get('url','')}_{d.metadata.get('chunk_idx', 0)}_{run_utc}"
    return md5(base.encode("utf-8")).hexdigest()

def chunks(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]


## Label작업 수행

In [38]:
for ticker in Custer_Having_ticker_lst:
    print(f"========= ticker {ticker} 진행 중~ =============")
    task_1_df = get_recent_articles(cusA_news_df,ticker=ticker,days = 30)
    task_1_df['Date'] = task_1_df.Date.astype('str')
    # document 만들기
    task_1_docs = make_documents(task_1_df)
    length_docs = len(task_1_docs)
    print(f'각 {ticker} 별 docs의 개수 : {length_docs}')
    BATCH = 64  # 상황에 맞게 조절

    for docs in tqdm(chunks(task_1_docs, BATCH), total=(len(task_1_docs) + BATCH - 1) // BATCH):

        ids = [make_doc_id(d) for d in docs]
        vectordb.add_documents(documents=docs, ids=ids)
        time.sleep(0.1) 

========= ticker 우리금융지주 진행 중~ =============


0it [00:00, ?it/s]

115it [02:56,  1.54s/it]


각 우리금융지주 별 docs의 개수 : 219


100%|██████████| 4/4 [00:10<00:00,  2.50s/it]


...

각 현대글로비스 별 docs의 개수 : 63


100%|██████████| 1/1 [00:03<00:00,  3.01s/it]


========= ticker 현대차 진행 중~ =============


303it [08:18,  1.65s/it]


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-3.5-turbo in organization org-CuTNQyXsUJcPWkwA97y1gHEH on tokens per min (TPM): Limit 200000, Used 194032, Requested 6075. Please try again in 32ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}

In [37]:
print("Number of documents in DB:", vectordb._collection.count())

Number of documents in DB: 685


## 7. cross-encoder 모델(w/langchain 예시)

Re-ranker 사용하기 전

In [39]:
retriver_target = ['우리금융지주','삼성화재','카카오뱅크','삼성생명','메리츠금융지주','현대모비스','현대글로비스','SK하이닉스','현대오토에버']

In [40]:
retriever_total = vectordb.as_retriever(search_kwargs={"k": 50,'filter' : {'ticker' : retriver_target}})

In [41]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i + 1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

Re-ranker 모델 사용 후

In [43]:
#model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
compressor = CrossEncoderReranker(model=model, top_n=50)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever_total
)

검색 유사도 측정
 - 이유 : 적정한 조건을 찾기 위함

In [47]:
scored_docs = []

retriever가 내부에서 메타데이터 필터(where/filter) 를 쓰고 있는데, 거기에 ['우리금융지주', ...] 같은 리스트를 그대로 넣어둔 상태예요. 대부분의 벡터스토어(특히 Chroma/LangChain)는 where/filter 값이 단일 값(str/int/float) 이거나 연산자 표현식이어야 하고, 리스트는 $in 같은 연산자로 감싸줘야 합니다. 그래서 get_relevant_documents() 부를 때마다 같은 잘못된 필터가 적용되어 터진 거죠.

빠른 해결책 (권장)
루프마다 단일 종목으로 필터를 바꿔서 조회하세요.

In [48]:
for ticker in retriver_target:
    retriever = vectordb.as_retriever(search_kwargs={"k": 50,'filter' : {'ticker' : ticker}})
    CrossEncoder_prompt = f'''
    이 뉴스들 중에서 "{ticker}"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, {ticker}가 단순히 함께 언급된 기사라면 제외해줘.
    '''
    print(CrossEncoder_prompt.strip())
    raw_docs = retriever.get_relevant_documents(CrossEncoder_prompt)
    pairs = [(CrossEncoder_prompt, d.page_content) for d in raw_docs]
    scores = model.score(pairs)

    # 3) 점수 붙이고 재정렬
    for d, s in zip(raw_docs, scores):
        dd = deepcopy(d)
        dd.metadata["relevance_score"] = float(s)
        scored_docs.append(dd)
    #scored_docs.sort(key=lambda x: (x.metadata[''],x.metadata["relevance_score"]), reverse=True)

이 뉴스들 중에서 "우리금융지주"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, 우리금융지주가 단순히 함께 언급된 기사라면 제외해줘.


/var/folders/xq/zzsj9f116r7brgm2wm610nx00000gn/T/ipykernel_36208/2935046578.py:8: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  raw_docs = retriever.get_relevant_documents(CrossEncoder_prompt)
/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


이 뉴스들 중에서 "삼성화재"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, 삼성화재가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


이 뉴스들 중에서 "카카오뱅크"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, 카카오뱅크가 단순히 함께 언급된 기사라면 제외해줘.


...

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


이 뉴스들 중에서 "SK하이닉스"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, SK하이닉스가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


이 뉴스들 중에서 "현대오토에버"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, 현대오토에버가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [49]:
# 4) 확인 - raw data
for i, d in enumerate(scored_docs[:50], 1):
    print(f"{i:02d} | original_idx : {d.metadata['original_idx']} | chunk_idx : {d.metadata['chunk_idx']}| score : {d.metadata['relevance_score']:.4f} | {d.metadata.get('title')}")

01 | original_idx : 245 | chunk_idx : 1| score : 0.1130 | 더 커진 주주환원 기대감…금융주, 어닝시즌 관전포인트
02 | original_idx : 158 | chunk_idx : 0| score : 0.0100 | 대출 수익성 악화에…4대 금융 실적 꺾였다
03 | original_idx : 475 | chunk_idx : 0| score : 0.2153 | 4대금융 2분기 순익 5.4조…사상 최대 실적
04 | original_idx : 237 | chunk_idx : 0| score : 0.6022 | '우리금융지주' 52주 신고가 경신, 전일 외국인 대량 순매수
05 | original_idx : 245 | chunk_idx : 3| score : 0.0047 | 더 커진 주주환원 기대감…금융주, 어닝시즌 관전포인트
06 | original_idx : 158 | chunk_idx : 1| score : 0.3583 | 대출 수익성 악화에…4대 금융 실적 꺾였다
07 | original_idx : 1 | chunk_idx : 0| score : 0.1089 | "실적·주주환원 훈풍"…은행·증권주 신고가 행진, 스탁론 매수세도 유입
08 | original_idx : 23 | chunk_idx : 0| score : 0.2479 | '우리금융지주' 52주 신고가 경신, 갈수록 돋보일 고배당 매력 - NH투자증권, BUY
09 | original_idx : 258 | chunk_idx : 2| score : 0.0045 | 4대금융, 이자 대신 환차익 덕 봤다…"하반기엔 불투명"
10 | original_idx : 229 | chunk_idx : 0| score : 0.0163 | 금융지주 영구채 '큰장' 선다…신한·하나 등 1.8조원 쏟아질 듯
11 | original_idx : 476 | chunk_idx : 0| score : 0.3970 | "우리금융지주, 연말로 갈수록 고배당 부각…목표가↑"-NH
12 |

### Re-Ranker 모델 측정

In [50]:
rerank_df = pd.DataFrame(list(map(lambda x: x.metadata,scored_docs)))

In [51]:
rerank_df.ticker.value_counts()

ticker
우리금융지주     50
삼성화재       50
카카오뱅크      50
삼성생명       50
메리츠금융지주    50
현대모비스      50
현대글로비스     50
SK하이닉스     50
현대오토에버     22
Name: count, dtype: int64

In [52]:
rerank_df.head()

,chunk_idx,title,label,original_idx,ticker,url,Date,relevance_score
0,1,"더 커진 주주환원 기대감…금융주, 어닝시즌 관전포인트",1,245,우리금융지주,https://www.hankyung.com/article/2025070981175,2025-07-09,0.113008
1,0,대출 수익성 악화에…4대 금융 실적 꺾였다,1,158,우리금융지주,https://www.hankyung.com/article/2025071501731,2025-07-15,0.010040
2,0,4대금융 2분기 순익 5.4조…사상 최대 실적,1,475,우리금융지주,https://www.hankyung.com/article/2025072528861,2025-07-25,0.215311
3,0,"'우리금융지주' 52주 신고가 경신, 전일 외국인 대량 순매수",1,237,우리금융지주,https://www.hankyung.com/article/202507145874L,2025-07-14,0.602175
4,3,"더 커진 주주환원 기대감…금융주, 어닝시즌 관전포인트",1,245,우리금융지주,https://www.hankyung.com/article/2025070981175,2025-07-09,0.004671


In [53]:
rerank_df= rerank_df.sort_values(by=['ticker','relevance_score'])

In [54]:
rerank_df.shape

(422, 8)

In [56]:
rerank_df.ticker.value_counts()

ticker
SK하이닉스     50
메리츠금융지주    50
삼성생명       50
삼성화재       50
우리금융지주     50
카카오뱅크      50
현대글로비스     50
현대모비스      50
현대오토에버     22
Name: count, dtype: int64

In [131]:
rerank_df[rerank_df.ticker=='SK하이닉스'].label.value_counts()

label
1    49
0     1
Name: count, dtype: int64

In [57]:
VecDB = vectordb._collection
total_results = VecDB.get(
         include = ['documents','metadatas']   
        )

In [58]:
total_df = pd.DataFrame(total_results['metadatas'])

In [59]:
total_df.shape

(1904, 7)

In [60]:
total_df.ticker.value_counts()

ticker
SK하이닉스     968
우리금융지주     219
삼성생명       193
현대모비스      128
삼성화재       123
카카오뱅크      122
메리츠금융지주     66
현대글로비스      63
현대오토에버      22
Name: count, dtype: int64

In [132]:
total_df[total_df.ticker=='SK하이닉스'].label.value_counts()

label
1    797
0    171
Name: count, dtype: int64

In [61]:
import numpy as np
import pandas as pd

def _dcg_at_k(labels, k=50):
    L = np.asarray(labels)[:k]
    if L.size == 0: return 0.0
    discounts = 1.0 / np.log2(np.arange(2, L.size + 2))
    return float(np.sum(L * discounts))

def ndcg_at_k(labels, k=50):
    labels = np.asarray(labels)
    dcg  = _dcg_at_k(labels, k)
    idcg = _dcg_at_k(np.sort(labels)[::-1], k)
    return 0.0 if idcg == 0 else dcg / idcg

def precision_at_k(labels, k=50):
    L = np.asarray(labels)[:k]
    return float(L.mean()) if L.size else 0.0

def recall_at_k(labels, total_relevant, k=50):
    if not total_relevant or total_relevant <= 0: return 0.0
    return float(np.sum(np.asarray(labels)[:k])) / float(total_relevant)

def mrr_at_k(labels, k=50):
    L = np.asarray(labels)[:k]
    hit = np.where(L > 0)[0]
    return 0.0 if hit.size == 0 else 1.0 / (hit[0] + 1)

def map_at_k(labels, total_relevant, k=50):
    L = np.asarray(labels)[:k]
    if L.sum() == 0: return 0.0
    precisions, hit = [], 0
    for i, y in enumerate(L, start=1):
        if y:
            hit += 1
            precisions.append(hit / i)
    denom = max(1, min(total_relevant if total_relevant is not None else int(L.sum()), k))
    return float(np.sum(precisions) / denom)

def eval_reranker_chunk(pool_df, ranked_df, k=50, rank_col="rank", score_col="relevance_score"):
    # 1) 풀: label만 필요 (score/rank 불필요)
    total_rel_pool = int(
        pd.to_numeric(pool_df["label"], errors="coerce").fillna(0).clip(0,1).sum()
    )
    print(f'total_rel_pool : {total_rel_pool}')

    # 2) 랭크드: 순서가 필요 (rank 우선, 없으면 score로 정렬, 둘 다 없으면 현재 순서 사용)
    g = ranked_df.copy()
    if rank_col in g.columns:
        g = g.sort_values(rank_col, ascending=True)
    elif score_col in g.columns:
        g = g.sort_values(score_col, ascending=False)
        g[rank_col] = np.arange(1, len(g)+1)
    else:
        g[rank_col] = np.arange(1, len(g)+1)

    y_topk = pd.to_numeric(g["label"], errors="coerce").fillna(0).clip(0,1).astype(int).values[:k]
    print(f'y_topk : {y_topk}')
    return {
        f"Precision@{k}": precision_at_k(y_topk, k),
        f"Recall@{k}(pool)": recall_at_k(y_topk, total_rel_pool, k),
        f"MRR@{k}": mrr_at_k(y_topk, k),
        f"MAP@{k}": map_at_k(y_topk, total_rel_pool, k),
        f"nDCG@{k}": ndcg_at_k(y_topk, k),
        "PoolSize": int(len(pool_df)),
        "PoolRelevant": total_rel_pool,
        "TopK": int(min(len(g), k)),
        "TopKRelevant": int(y_topk.sum()),
    }


In [66]:
rerank_eval = []

In [67]:
for kind in retriver_target:
    total_df_target = total_df[total_df.ticker==kind]
    rerank_df_target = rerank_df[rerank_df.ticker==kind]
    rerank_eval.append(eval_reranker_chunk(total_df_target,rerank_df_target))

total_rel_pool : 180
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 91
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 0 1 1 1 0 0 0 1 1 1 1 1 0 1 1
 1 1 1 1 1 0 0 1 1 0 1 0 0]
total_rel_pool : 99
y_topk : [1 1 1 1 1 1 1 1 1 1 0 1 1 1 0 1 1 1 1 0 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 0
 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 109
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 0 0
 1 1 1 1 0 0 1 0 0 1 1 0 0]
total_rel_pool : 47
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 0 0 1 1
 0 0 0 0 1 0 0 0 0 0 0 0 0]
total_rel_pool : 98
y_topk : [1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 1 1 1 1 1 0 1 1 0
 1 1 1 1 1 0 1 0 1 1 0 0 1]
total_rel_pool : 49
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 1 1 0 0 0 1 1 1 0 1 1 1 1 1 1 0 0 1
 1 1 1 1 1 1 1 1 0 1 0 1 1]
total_rel_pool : 797
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1

In [68]:
rerank_eval_con = dict(zip(Custer_Having_ticker_lst,rerank_eval))

In [69]:
pd.DataFrame(rerank_eval_con)

,우리금융지주,삼성화재,삼성생명,카카오뱅크,메리츠금융지주,현대모비스,현대오토에버,SK하이닉스,현대글로비스
Precision@50,1.000000,0.780000,0.900000,0.800000,0.680000,0.820000,0.800000,0.980000,0.636364
Recall@50(pool),0.277778,0.428571,0.454545,0.366972,0.723404,0.418367,0.816327,0.061481,1.000000
MRR@50,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
MAP@50,1.000000,0.718375,0.817043,0.764605,0.694256,0.740508,0.729028,0.973507,0.896701
nDCG@50,1.000000,0.982615,0.977576,0.990670,0.991350,0.964739,0.976200,0.998768,0.972676
PoolSize,219.000000,123.000000,122.000000,193.000000,66.000000,128.000000,63.000000,968.000000,22.000000
PoolRelevant,180.000000,91.000000,99.000000,109.000000,47.000000,98.000000,49.000000,797.000000,14.000000
TopK,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,22.000000
TopKRelevant,50.000000,39.000000,45.000000,40.000000,34.000000,41.000000,40.000000,49.000000,14.000000


### original 본문 가져오기

In [91]:
test_ticker = 'SK하이닉스'

In [92]:
CrossEncoder_prompt_test = f'''
이 뉴스들 중에서 "{test_ticker}"가 핵심 주제로 다뤄진 기사만 알려줘.
다른 회사 언급이 많거나, {test_ticker}가 단순히 함께 언급된 기사라면 제외해줘.
'''

In [ ]:
# rerank을 통해 문서 가져오기
reranked_docs = compressor.compress_documents(raw_docs, query=CrossEncoder_prompt_test)

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [105]:
retriever_test = vectordb.as_retriever(search_kwargs={"k": 50,'filter' : {'ticker' : test_ticker}})

In [106]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever_test
)

In [107]:
compressed_docs = compression_retriever.invoke(CrossEncoder_prompt)

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [108]:
def get_full_article_from_chroma(original_idx: int, kind : str, vectordb) -> dict:
    """original_idx 기준으로 chunk들을 모아 원문 복원"""
    # 1. 해당 article의 모든 chunk 가져오기
    VecDB = vectordb._collection
    results = VecDB.get(
      where = { "$and" : [ # 빈 쿼리로 전체 탐색
            {"original_idx": original_idx}, 
            {"ticker": kind}
            ]
         },
         include = ['documents','metadatas']   
        )
        
    if not results:
        return {"error": f"No chunks found for original_idx {original_idx}"}

    #print(results)
    #print(results['metadatas'][0]['chunk_idx'])

    # 4. 대표 metadata 하나 뽑아 저장
    return {
        "title": results['metadatas'][0]["title"],
        "url": results['metadatas'][0]["url"],
        "Date": results['metadatas'][0]["Date"],
        "ticker": results['metadatas'][0]["ticker"],
        "content": results['documents'][0]
    }

In [109]:
original_idxs = list(map(lambda x: x.metadata['original_idx'], compressed_docs))

In [110]:
original_idxs

[3739,
 3607,
 3798,
 3838,
 3791,
 3706,
 3951,
 3659,
 3885,
 3666,
 3538,
 3897,
 3583,
 3980,
 3511,
 3691,
 3806,
 3531,
 3802,
 3966,
 3618,
 3656,
 3690,
 3844,
 3986,
 3732,
 3584,
 3764,
 3537,
 3802,
 3882,
 3955,
 3588,
 3700,
 3801,
 3802,
 3808,
 3648,
 3922,
 3529,
 3605,
 3785,
 3700,
 3802,
 3987,
 3847,
 3802,
 3763,
 3905,
 3807]

In [112]:
get_full_article_from_chroma(3980, kind=test_ticker,vectordb=vectordb)

{'title': '05일, 외국인 거래소에서 SK하이닉스(+2.13%), 삼성SDI(+10.22%) 등 순매수',
 'url': 'https://www.hankyung.com/article/202508053349L',
 'Date': '2025-08-05',
 'ticker': 'SK하이닉스',
 'content': '외국인 투자자는 05일 거래소에서 SK하이닉스, 삼성SDI, SK바이오팜 등을 중점적으로 사들인 것으로 나타났다.외국인 투자자의 순매수 상위 20개 종목은 SK하이닉스, 삼성SDI, SK바이오팜, 삼성중공업, 두산에너빌리티, 한화오션, 에코프로머티, LS ELECTRIC, 두산, 한국전력등이다.이중에 전기,전자 업종에 속한 종목이 6개 포함되어 있다.이날 외국인 투자자가 순매수한 종목들 중에 SK하이닉스, 삼성SDI, SK바이오팜 등은 전일 대비 주가가 상승했다.[08월05일]거래소 외국인 순매수 상위 종목한경로보뉴스이 기사는 한국경제신문과 금융 AI 전문기업 씽크풀이 공동 개발한 기사 자동생성 알고리즘에 의해 실시간으로 작성된 것입니다.'}

In [113]:
top_n_original = [get_full_article_from_chroma(idx,kind=test_ticker,vectordb=vectordb) for idx in original_idxs]

In [114]:
top_n_original[0]

{'title': '24일, 외국인 거래소에서 한화오션(+6.35%), SK하이닉스(+0.19%) 등 순매수',
 'url': 'https://www.hankyung.com/article/202507240571L',
 'Date': '2025-07-24',
 'ticker': 'SK하이닉스',
 'content': '외국인 투자자는 24일 거래소에서 한화오션, SK하이닉스, 두산에너빌리티 등을 중점적으로 사들인 것으로 나타났다.외국인 투자자의 순매수 상위 20개 종목은 한화오션, SK하이닉스, 두산에너빌리티, LG에너지솔루션, 기아, 한화비전, 한화에어로스페이스, 삼성전자, 삼성바이오로직스, LG화학등이다.이중에 전기,전자 업종에 속한 종목이 6개 포함되어 있다.한화오션, SK하이닉스, 두산에너빌리티 등은 전일 대비 주가가 상승했고, 기아, 삼성전자, 현대차 등은 주가가 하락했다.[07월24일]거래소 외국인 순매수 상위 종목한경로보뉴스이 기사는 한국경제신문과 금융 AI 전문기업 씽크풀이 공동 개발한 기사 자동생성 알고리즘에 의해 실시간으로 작성된 것입니다.'}

In [115]:
original_contents = list(map(lambda x: x['content'],top_n_original))

In [116]:
total_contents = ''.join(original_contents)

In [117]:
total_contents

'외국인 투자자는 24일 거래소에서 한화오션, SK하이닉스, 두산에너빌리티 등을 중점적으로 사들인 것으로 나타났다.외국인 투자자의 순매수 상위 20개 종목은 한화오션, SK하이닉스, 두산에너빌리티, LG에너지솔루션, 기아, 한화비전, 한화에어로스페이스, 삼성전자, 삼성바이오로직스, LG화학등이다.이중에 전기,전자 업종에 속한 종목이 6개 포함되어 있다.한화오션, SK하이닉스, 두산에너빌리티 등은 전일 대비 주가가 상승했고, 기아, 삼성전자, 현대차 등은 주가가 하락했다.[07월24일]거래소 외국인 순매수 상위 종목한경로보뉴스이 기사는 한국경제신문과 금융 AI 전문기업 씽크풀이 공동 개발한 기사 자동생성 알고리즘에 의해 실시간으로 작성된 것입니다.기관 투자자는 30일 거래소에서 삼성전자, 현대차, 삼성전기 등을 중점적으로 사들인 것으로 나타났다.기관 투자자의 순매수 상위 20개 종목은 삼성전자, 현대차, 삼성전기, KODEX 레버리지, 삼성SDI, SK하이닉스, 한화에어로스페이스, 카카오, LG에너지솔루션, 기아등이다.이중에 전기,전자 업종에 속한 종목이 8개 포함되어 있다.삼성전자, 현대차, 삼성전기 등은 전일 대비 주가가 상승했고, 한화에어로스페이스, 한국전력 등은 주가가 하락했다.[07월30일]거래소 기관 순매수 상위 종목한경로보뉴스이 기사는 한국경제신문과 금융 AI 전문기업 씽크풀이 공동 개발한 기사 자동생성 알고리즘에 의해 실시간으로 작성된 것입니다.외국인 투자자는 17일 거래소에서 SK하이닉스, 카카오페이, NAVER 등을 중점적으로 매도한 것으로 나타났다.외국인 투자자의 순매도 상위 20개 종목은 SK하이닉스, 카카오페이, NAVER, 두산에너빌리티, HD현대미포, 삼성에스디에스, 하나금융지주, HD현대일렉트릭, 현대차, 풍산등이다.이중에 운수장비 업종에 속한 종목이 4개 포함되어 있다.HD현대미포, 하나금융지주, 현대차 등은 전일 대비 주가가 상승했고, SK하이닉스, 카카오페이, NAVER 등은 주가가 하락했다.[07월17일]거래소 외국인 순매도 

In [118]:
len(total_contents)

24304

## 요약하기

### 요약함수 호출
- 가져온 원 본문을 전부 적용하기

In [119]:
from summart_function_openai_2 import NewsSummaryAgent, build_summary_graph  # 너가 만든 것

In [124]:
def summarize_top_articles_2(total_contents: str,ticker:str,max_iters:int) -> pd.DataFrame:
    agent = NewsSummaryAgent(max_iters=max_iters)
    runnable = build_summary_graph(agent)

    rows = []
    doc = total_contents

    state = {"article": doc, "summary": "", "feedback": "", "iteration": 0}
    result = runnable.invoke(state)

    print("✅ 실행 결과 키:", result.keys())
    # 여기서 final_summary 반드시 존재해야 함(위 패치 기준)
    final_summary = result.get("final_summary")
    if not final_summary:
        print("❌ final_summary 없음. 디버그용 전체 상태:", result)
        # 계속 진행할지, 실패로 표기할지 선택

    rows.append({
        "ticker": ticker,
        "date": '2025-07-30', # 위 조회 기준일자로 연동시켜서 바꿀 예정
        "summary": final_summary,
        "feedback": result.get("last_feedback", "피드백 없음"),
    })

    return pd.DataFrame(rows)


In [125]:
result_df = summarize_top_articles_2(total_contents,ticker=test_ticker,max_iters=5)

summart : ✅ 주요 요약
- 외국인 투자자, 특정 종목 집중 매수 및 매도
  외국인 투자자는 한화오션, SK하이닉스, 두산에너빌리티 등을 중점적으로 매수했으며, SK하이닉스, 카카오페이, NAVER 등을 매도한 것으로 나타났다.

- 기관 투자자, 전기·전자 업종에 집중
  기관 투자자는 삼성전자, 현대차, 삼성전기 등 전기·전자 업종의 종목을 중점적으로 매수했으며, SK하이닉스, KODEX 200선물인버스2X 등을 매도했다.

- 코스피지수 상승, 투자자 고민 증가
  코스피지수가 3200을 넘어섰지만, 주식 매입 시점을 고민하는 투자자들이 증가하고 있으며, 기존 대장주들의 추가 상승 가능성과 부담감이 동시에 작용하고 있다.

🔑 키워드
- 외국인 투자자
- 기관 투자자
- SK하이닉스
- 삼성전자
- 코스피지수
- 전기·전자 업종
[should_stop] Iteration: 0
[should_stop] Feedback:
 - 정확성: 부족함 <reason> [요약이 원문 기사에서 다루어진 모든 세부 사항을 정확하게 반영하지 못하고 있습니다. 예를 들어, 외국인과 기관 투자자의 매수 및 매도 종목에 대한 구체적인 정보가 일부 누락되었습니다.] </reason>
- 포괄성: 부족함 <reason> [기사의 주요 내용 중 일부가 요약에 포함되지 않았습니다. 예를 들어, 특정 날짜에 대한 투자자 행동이나 주가 변동에 대한 세부 정보가 누락되었습니다.] </reason>
- 간결성: 좋음 <reason> [요약은 불필요한 표현 없이 간결하게 작성되었습니다.] </reason>
- 문장구성: 좋음 <reason> [문장이 자연스럽고 명확하게 구성되어 있습니다.] </reason>
- 일관성: 부족함 <reason> [특정 종목에 대한 내용만 존재하는 것이 아니라 다양한 종목과 투자자 행동에 대한 정보가 포함되어 있습니다.] </reason>

**피드백:**
요약의 정확성과 포괄성을 개선하기 위해 원문 기사에서 다루어진 모든 주요 세부 사항을 포함하도록 해야

In [126]:
result_df

,ticker,date,summary,feedback
0,SK하이닉스,2025-07-30,"✅ 주요 요약\n\n- **외국인 투자자, 전기·전자 및 운수장비 업종에 집중 매수...",**평가:**\n\n- 정확성: 부족함 <reason> [요약에서 언급된 일부 내용...


In [127]:
print(result_df.summary.values[0])

✅ 주요 요약

- **외국인 투자자, 전기·전자 및 운수장비 업종에 집중 매수 및 매도**  
  외국인 투자자들은 한화오션, SK하이닉스, 두산에너빌리티 등을 중점적으로 매수했으며, SK하이닉스, 카카오페이, NAVER 등을 매도했습니다. 특히, 전기·전자 업종에 속한 종목들이 다수 포함되어 있으며, 이들 종목의 주가 변동이 투자 전략에 영향을 미쳤습니다.

- **기관 투자자, 전기·전자 및 운수장비 업종에 집중**  
  기관 투자자들은 삼성전자, 현대차, 삼성전기 등 전기·전자 업종의 종목을 중점적으로 매수했으며, SK하이닉스, KODEX 200선물인버스2X 등을 매도했습니다. 이들 종목 중 일부는 전일 대비 주가가 상승했습니다.

- **코스피지수 상승, 투자자 고민 증가**  
  코스피지수가 3200을 넘어섰지만, 주식 매입 시점을 고민하는 투자자들이 증가하고 있습니다. 기존 대장주들의 추가 상승 가능성과 이미 충분히 올랐다는 부담감이 동시에 작용하고 있습니다.

- **주요 종목의 주가 변동**  
  한화오션, SK하이닉스, 두산에너빌리티 등은 전일 대비 주가가 상승했으며, 기아, 삼성전자, 현대차 등은 주가가 하락했습니다. 이러한 변동은 투자자들의 매수 및 매도 전략에 영향을 미치고 있습니다.

🔑 키워드
- 외국인 투자자
- 기관 투자자
- SK하이닉스
- 삼성전자
- 코스피지수
- 전기·전자 업종

**개선 포인트:**
- 외국인 및 기관 투자자의 매수 및 매도 종목에 대한 구체적인 정보를 포함하여 정확성을 높였습니다.
- 주가 변동에 대한 세부 사항을 추가하여 포괄성을 강화했습니다.
- 다양한 종목과 투자자 행동을 일관성 있게 다루었습니다.


## 피드백 정리
- 삼성전자  : 삼성전자에 대한 내용 잘 요약
- 우리금융지주 : 우리금융지주에 대한 내용 잘 요약
- SK하이닉스 : 요약은 되었으나,, 시장 내/외부 사람들의 구매 패턴, 동향에 대한 정보가 주로 요약이 되고 있음<br>
            > recall@50(pool) 이 상당히 낮게 나옴!
  > SK하이닉스에 대한 케이스를 봤을 때, chunk에 대해서만 요약을 해야하나 싶음<br>
  > 2025.08.11) 현재 작업은 전체 원문에 대한 요약이었음. 그래서.. SK 하이닉스 회사 본질에 대한 정보를 제대로 요약해주지 못하나 싶음<br>
  > 프롬프트 내부 요약 평가 기준 의 "일관성" 영역에 명확히 종목명을 말해야겠다..

# 과거 버전

In [204]:
import pandas as pd
from datetime import datetime, timedelta
# cross-encoder
from sentence_transformers import CrossEncoder
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from summary_function_openai import NewsSummaryAgent, build_summary_graph  # 너가 만든 것

# 1. 모델 준비 (CrossEncoder for Re-ranking) # 예시 모델 하나 생성
rerank_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# 2. 최근 5일치 필터링 함수
def get_recent_articles(df: pd.DataFrame, ticker: str, days: int = 5):
    df['Date'] = pd.to_datetime(df['Date'])
    today = df['Date'].max()
    recent_df = df[
        (df['ticker'] == ticker) &
        (df['Date'] >= today - timedelta(days=days))
    ]
    return recent_df.sort_values(by="Date", ascending=False)

# 3. huggingface 모델 이용.
def rerank_articles(df: pd.DataFrame,ticker:str ,query: str, top_k: int = 5):
    task_df= df[df.ticker == ticker]
    docs = task_df['content'].tolist()
    # 모델 이용?? 
    pairs = [(query, doc) for doc in docs]
    scores = rerank_model.predict(pairs)
    
    task_df_2 = task_df.copy()
    task_df_2['score'] = scores
    return task_df_2.sort_values(by='score', ascending=False).head(top_k)

# 4. 전체 요약 실행 함수
def summarize_top_articles(df: pd.DataFrame, ticker: str, query: str, top_k: int = 5):
    recent_df = get_recent_articles(df, ticker)
    top_df = rerank_articles(recent_df, query=query,ticker=ticker, top_k=top_k)

    agent = NewsSummaryAgent()
    runnable = build_summary_graph(agent)

    results = []
    for _, row in top_df.iterrows():
        state = {
            "article": row['content'],
            "summary": "",
            "feedback": "",
            "iteration": 0
        }
        result = runnable.invoke(state)


        print("✅ 실행 결과 타입:", type(result))
        print("✅ 실행 결과 키 목록:", result.keys())
        print("✅ 실행 결과 전체 내용:", result)

        if "final_summary" not in result:
            print("❌ final_summary 키가 없습니다. 중단합니다.")
            continue  # 또는 raise Exception("final_summary 없음")

        print(f'실행 결과 : {result}')
        results.append({
            "ticker": row['ticker'],
            "date": row['Date'],
            "header": row['header'],
            "url": row['url'],
            "summary": result["final_summary"], 
             "feedback": result.get("last_feedback", "피드백 없음")
        })
    return pd.DataFrame(results)


In [55]:
query = "우리금융지주 관련 시황"
ticker = "우리금융지주"  # 예시
result_df = summarize_top_articles(cusA_news_df, ticker=ticker, query=query, top_k=3)

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


[should_stop] Iteration: 1
[should_stop] Feedback:
 - 정확성: 좋음 <reason> 원문의 내용을 정확하게 반영하고 있음. </reason>
- 포괄성: 부족함 <reason> 원문에서 언급된 '기술 경쟁력 확보, 고객사 확대, 수익구조 개선 등 실질적 사업성과로 뒷받침돼야 한다는 지적' 등의 중요한 내용이 누락되었음. </reason>
- 간결성: 좋음 <reason> 불필요한 표현 없이 요약 내용을 간결하게 전달하였음. </reason>
- 문장구성: 좋음 <reason> 문장이 자연스럽고 명확하게 구성되어 있음. </reason>

[피드백]
요약의 포괄성이 부족한 점이 아쉽습니다. 원문에서 언급된 '기술 경쟁력 확보, 고객사 확대, 수익구조 개선 등 실질적 사업성과로 뒷받침돼야 한다는 지적' 등의 중요한 내용을 요약에 포함시키면 더욱 완벽한 요약이 될 것 같습니다. 이 부분을 고려하여 요약을 수정해보시는 것을 추천드립니다.
✅ '정확성' 평가 통과
❌ '포괄성' 평가에서 좋음이 아님
[should_stop] next_step = no


KeyError: 'Input to PromptTemplate is missing variables {\'"foo"\', \'"properties"\'}.  Expected: [\'"foo"\', \'"properties"\', \'article\', \'feedback\', \'summary\'] Received: [\'article\', \'summary\', \'feedback\']\nNote: if you intended {"foo"} to be part of the string and not a variable, please escape it with double curly braces like: \'{{"foo"}}\'.\nFor troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_PROMPT_INPUT '

In [35]:
def summarize_top_articles(total_contents :str):

    agent = NewsSummaryAgent()
    runnable = build_summary_graph(agent)
    results = []
    
    doc = total_contents

    state = {
        "article": doc['content'],
        "summary": "",
        "feedback": "",
        "iteration": 0
    }
    result = runnable.invoke(state)


    print("✅ 실행 결과 타입:", type(result))
    print("✅ 실행 결과 키 목록:", result.keys())
    print("✅ 실행 결과 전체 내용:", result)

    if "final_summary" not in result:
        print("❌ final_summary 키가 없습니다.")

    print(f'실행 결과 : {result}')
    results.append({
        "ticker": doc['ticker'],
        "date": doc['Date'],
        "header": doc['title'],
        "url": doc['url'],
        "summary": result["final_summary"], 
            "feedback": result.get("last_feedback", "피드백 없음")
    })
    return pd.DataFrame(results)
